## 基于MindNLP的SAM模型应用开发

### 1 **环境配置：**

1. MindSpore 2.3.1
2. Mindnlp 0.4.0
3. Python 3.9.0
4. matplotlib

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"       #huggingface镜像      

也可以在华为云上测试项目代码运行： https://pangu.huaweicloud.com/gallery/asset-detail.html?id=de70ce94-406b-4625-972e-1a763c01d32c

### 2 SAM 模型
原文地址：https://arxiv.org/pdf/2408.03322

原代码链接：https://github.com/bowang-lab/medsam

#### 2.1 SAM 模型

SAM（Segment Anything Model）是一个创新的大规模预训练模型，用于图像分割，它通过ViT图像编码器、提示器和轻量级解码器的组合来实现精确的分割任务。这里我使用了MedSAM，而MedSAM是在SAM基础上发展起来的，专注于医学图像分割，具有更高的准确性和泛化能力。MedSAM在庞大的医学图像数据集上训练，涵盖了多种成像模态和癌症类型。使用MedSAM来识别乳腺癌，利用其强大的分割能力来辅助临床诊断和治疗计划的制定，从而提高乳腺癌检测的精确度和效率。


#### 2.2 导入模型到mindnlp
可以从中 `dir(mindnlp.transformers)` 查看和导入到关于VIT的包

In [ ]:
import mindnlp
# dir(mindnlp.transformers)   

#### 2.3 从HuggingFace加载模型到GPU
接下来，让我们加载SAM模型及其对应的处理器。模型可以在hub上找到，地址在这里：https://huggingface.co/wanglab/medsam-vit-base.

In [ ]:
import mindspore 
from mindnlp.transformers import SamModel, SamProcessor      
     
# 加载预训练的 SegFormer 模型
processor = SamProcessor.from_pretrained("wanglab/medsam-vit-base")
model = SamModel.from_pretrained("wanglab/medsam-vit-base")

#### 2.4 加载图像 + 真实分割映射图
接下来，让我们从MedSAM作者收集的数据集中加载一张图像及其对应的真实分割映射图。


In [ ]:
from datasets import load_dataset

dataset = load_dataset("nielsr/breast-cancer", split="train")

In [ ]:
import numpy as np
from PIL import Image

idx = 10

# 加载图片
image = dataset[idx]["image"]
image

这里，分割是一个二维的NumPy数组，其中用1表示感兴趣的区域，并进行可视化：

In [ ]:
ground_truth_seg = np.array(dataset[idx]["label"])
np.unique(ground_truth_seg)

In [ ]:
palette = [[120, 120, 120], [6, 230, 230]]

color_seg = np.zeros((ground_truth_seg.shape[0], ground_truth_seg.shape[1], 3), dtype=np.uint8) # height, width, 3

for label, color in enumerate(palette):
    color_seg[ground_truth_seg == label, :] = color

Image.fromarray(color_seg)

##### 2.5 加载边界框提示并进行预测
MedSAM的作者使用基于真实分割的边界框来提示模型。让我们在这里进行同样的操作：

In [ ]:
def get_bounding_box(ground_truth_map):
  y_indices, x_indices = np.where(ground_truth_map > 0)
  x_min, x_max = np.min(x_indices), np.max(x_indices)
  y_min, y_max = np.min(y_indices), np.max(y_indices)
  H, W = ground_truth_map.shape
  x_min = max(0, x_min - np.random.randint(0, 20))
  x_max = min(W, x_max + np.random.randint(0, 20))
  y_min = max(0, y_min - np.random.randint(0, 20))
  y_max = min(H, y_max + np.random.randint(0, 20))
  bbox = [x_min, y_min, x_max, y_max]

  return bbox

input_boxes = get_bounding_box(ground_truth_seg)
input_boxes

In [ ]:
import matplotlib.pyplot as plt

def show_box(box, ax):
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, edgecolor='green', facecolor=(0,0,0,0), lw=2))  

def show_boxes_on_image(raw_image, boxes):
    plt.figure(figsize=(10,10))
    plt.imshow(raw_image)
    for box in boxes:
      show_box(box, plt.gca())
    plt.axis('on')
    plt.show()

In [ ]:
show_boxes_on_image(image, [input_boxes])

模型被提示在边界框内生成一个分割掩码，我们可以为模型准备输入并执行前向传播。

In [ ]:
# 处理输入图像，返回MindSpore Tensor
inputs = processor(images=image, return_tensors="ms")  # 使用“ms”来生成MindSpore的tensor
for k,v in inputs.items():
  print(k,v.shape)

In [ ]:
outputs = model(**inputs, multimask_output=False)
outputs.pred_masks.shape

#### 2.6 进行可视化
请注意，MedSAM使用了自定义的[DiceWithSigmoid损失函数](https://github.com/bowang-lab/MedSAM/blob/66cf4799a9ab9a8e08428a5087e73fc21b2b61cd/train.py#L70)进行了微调，因此我们需要在这里应用相应的[后处理](https://github.com/bowang-lab/MedSAM/blob/66cf4799a9ab9a8e08428a5087e73fc21b2b61cd/MedSAM_Inference.py#L67)：


In [ ]:
import mindspore.ops as ops
import numpy as np

# 应用Sigmoid函数
sigmoid = ops.Sigmoid()
medsam_seg_prob = sigmoid(outputs.pred_masks.squeeze(1))

# 将输出转换为NumPy数组
medsam_seg_prob_np = medsam_seg_prob.asnumpy().squeeze()

# 将软掩码转换为硬掩码
medsam_seg = (medsam_seg_prob_np > 0.5).astype(np.uint8)


In [ ]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

fig, axes = plt.subplots()

axes.imshow(np.array(image))
show_mask(medsam_seg, axes)
axes.title.set_text(f"Predicted mask")
axes.axis("off")

与此相比的是真实分割：

In [ ]:
fig, axes = plt.subplots()

axes.imshow(np.array(image))
show_mask(ground_truth_seg, axes)
axes.title.set_text(f"Ground truth mask")
axes.axis("off")